# Exploratory Data Analysis (EDA) & Feature Engineering

## 1. Load the Clean Dataset

I loaded the cleaned dataset from Week 1 to begin exploratory analysis.

In [2]:
import pandas as pd
df = pd.read_csv('/Users/arkyaghosh/churn_project_data/processed/telco_clean.csv')
print(df.shape)
print(df.head())

(7032, 20)
   gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  Female              0     Yes         No       1           No   
1    Male              0      No         No      34          Yes   
2    Male              0      No         No       2          Yes   
3    Male              0      No         No      45           No   
4  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity OnlineBackup  \
0  No phone service             DSL             No          Yes   
1                No             DSL            Yes           No   
2                No             DSL            Yes          Yes   
3  No phone service             DSL            Yes           No   
4                No     Fiber optic             No           No   

  DeviceProtection TechSupport StreamingTV StreamingMovies        Contract  \
0               No          No          No              No  Month-to-month   
1              Yes   

## 2. EDA on Categorical Variables

I examined churn rates across key categorical features to understand which customer characteristics correlate most strongly with churn.

In [3]:
churn_by_contract = df.groupby('Contract')['Churn'].apply(lambda x: (x == "Yes").mean())
churn_by_internet_service = df.groupby('InternetService')['Churn'].apply(lambda x: (x == "Yes").mean())
churn_by_online_security = df.groupby('OnlineSecurity')['Churn'].apply(lambda x: (x == "Yes").mean())

### Key Findings

| Feature | High Churn | Low Churn |
|---------|-----------|----------|
| **Contract Type** | Month-to-month (42.7%) | Two year (2.8%) |
| **Internet Service** | Fiber optic (41.9%) | No service (7.4%) |
| **Online Security** | No security (41.8%) | With security (14.6%) |


**Business Insights:**

1. **Contract Length is Highly Predictive:** Month-to-month contracts have a churn rate 15x higher than two-year contracts. This suggests customers with flexible contracts are less committed.
2. **Fiber Optic Service Quality Concern:** Fiber optic customers churn at 6x the rate of those without internet service. This could indicate service reliability issues or poor value proposition for fiber customers.
3. **Online Security as a Retention Tool:** Customers without online security churn at about 3x the rate of those with it. This service appears to increase customer loyalty.

## 3. EDA on Numeric Variables

I compared numeric characteristics between customers who churned vs. those who stayed.

In [4]:
tenure_by_churn = df.groupby('Churn')['tenure'].mean()
monthly_charges_by_churn = df.groupby('Churn')['MonthlyCharges'].mean()
total_charges_by_churn = df.groupby('Churn')['TotalCharges'].mean()


### Key Findings

| Metric | Churned | Retained | Difference |
|--------|---------|----------|------------|
| Average Tenure | 18 months | 38 months | -53% |
| Monthly Charges | \$74.44 | \$61.31 | +21% |
| Total Charges | \$1,532 | \$2,555 | -40% |


**Business Insights:**

1. **New Customers at Risk:** Customers who stayed had much longer tenure—about double that of those who churned. People who stay longer develop loyalty and commitment.
2. **Higher Pricing Drives Churn:** Churners pay 21% more per month than retained customers ($74.44 vs. $61.31). Pricing is definitely a factor in churn rates.
3. **Lower Lifetime Value:** While churners pay more monthly, they pay 40% less total charges due to shorter tenure.

# 3. Feature Engineering & Encoding

I applied two encoding techniques:

1. **Label Encoding** for binary variables (Yes/No, Female/Male) → convert to 0/1
2. **One-Hot Encoding** for multi-category variables → create separate binary columns

In [5]:
multiCategory = []
for i in df.columns:
    if len(df[i].unique()) == 2 and df[i].unique()[0] == "Yes" and df[i].unique()[1] == "No" or len(df[i].unique()) == 2 and df[i].unique()[0] == "No" and df[i].unique()[1] == "Yes":
        df[i] = df[i].map({'Yes': 1, 'No': 0})
    elif len(df[i].unique()) == 2 and df[i].unique()[0] == "Female" and df[i].unique()[1] == "Male" or len(df[i].unique()) == 2 and df[i].unique()[0] == "Male" and df[i].unique()[1] == "Female":
        df[i] = df[i].map({'Female': 1, 'Male': 0})
    elif len(df[i].unique()) > 2 and df[i].dtype == 'str':
        multiCategory.append(i)
df = pd.get_dummies(df, columns=multiCategory, drop_first=True, dtype=int)

### Why This Approach?

- **Label Encoding (0/1)** for categorical data with 2 clear options like "yes" or "no," label encoding works well–just convert 0 and 1.
- **One-Hot Encoding** for multi-category data, creates separate binary columns so the model doesn't incorrectly assume order or hierarchy between categories.
- **drop_first=True** removes one dummy column per feature to avoid multicollinearity, which is a statistical issue where redundant information can confuse the model's ability to learn.

### Result

The dataset now has **26 numeric columns** ready for logistic regression modeling, with the target variable `Churn` (0/1) ready for prediction.

---

## Summary

Week 2 revealed strong patterns in customer churn:
- **Contract type, internet service type, and add-on services** are powerful predictors
- **New customers and high-paying customers** are at elevated risk
- **Data is now encoded and ready** for logistic regression